# Setup


In [ ]:
import sys
from pathlib import Path

# import os
import importlib

sys.path.append(str(Path.cwd().parent.parent.parent.absolute()))

import config

importlib.reload(config)

import argparse
from context import Context
from utils.utils_chain import WrapperAddress
from tools.runners.farm_runner import (
    upgrade_farmv2_contracts,
    get_all_farm_v2_addresses,
    fetch_and_save_farms_from_chain,
    pause_farm_contracts,
    resume_farm_contracts,
)
from tools.runners.staking_runner import (
    upgrade_staking_contracts,
    get_staking_addresses_from_chain,
    pause_all_staking_contracts,
    resume_all_staking_contracts,
    fetch_and_save_stakings_from_chain,
)
from tools.runner import fetch_and_save_pause_state

# os.environ["MX_DEX_ENV"] = "shadowfork4"

context = Context()

farm_wasm = "https://github.com/multiversx/mx-exchange-sc/releases/download/v3.5.0-rc5/farm-with-locked-rewards.wasm"
farm_code_hash = "580f656492023a423177b40510ab1623a259367fc3bab19c86563524a32434b9"
staking_wasm = (
    "https://github.com/multiversx/mx-exchange-sc/releases/download/v3.5.0-rc5/farm-staking.wasm"
)
staking_code_hash = "25934cc11e7e50efa4f1dac6efe9efb462a30d9d33e4cc6071f38afb43adc17b"

Check owner balance for required minimum


In [ ]:
current_balance = context.network_provider.proxy.get_account(
    context.deployer_account.address
).balance
assert current_balance > 20 * 10**18, "Deployer account doesn't have enough balance"

Clean outputs folder


In [ ]:
import shutil

if config.UPGRADER_OUTPUT_FOLDER.exists():
    shutil.rmtree(config.UPGRADER_OUTPUT_FOLDER)

Prep contract pause states


In [ ]:
fetch_and_save_farms_from_chain("")

In [ ]:
fetch_and_save_stakings_from_chain("")

In [ ]:
fetch_and_save_pause_state("")

Pause contracts


In [ ]:
pause_farm_contracts("")

In [ ]:
pause_all_staking_contracts("")

# Upgrade farm contracts


In [ ]:
args = argparse.Namespace(bytecode=farm_wasm, compare_states=True)
upgrade_farmv2_contracts(args)

farm_addresses = get_all_farm_v2_addresses()
print(f"Checking {len(farm_addresses)} farm contracts for correct code hash...")
for address in farm_addresses:
    code_hash = context.network_provider.proxy.get_account(
        WrapperAddress(address)
    ).contract_code_hash.hex()
    assert code_hash == farm_code_hash, f"Code hash mismatch for {address}"
print("Done!")

# Upgrade staking contracts


In [ ]:
args = argparse.Namespace(bytecode=staking_wasm, compare_states=True, all=True)
upgrade_staking_contracts(args)

staking_addresses = get_staking_addresses_from_chain()
print(f"Checking {len(staking_addresses)} staking contracts for correct code hash...")
for address in staking_addresses:
    code_hash = context.network_provider.proxy.get_account(
        WrapperAddress(address)
    ).contract_code_hash.hex()
    assert code_hash == staking_code_hash, f"Code hash mismatch for {address}"
print("Done!")

# Resume contracts


In [ ]:
resume_farm_contracts("")

In [ ]:
resume_all_staking_contracts("")